# Configuration

### Importing Modules 
To import modules ad the same level in the file tree, we need to set the module path.

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    

### Cadquery Viewer Setup 

In [2]:
from cadquery import Workplane
from cadq_server import cq_helper

import cadquery as cq
from jupyter_cadquery import set_sidecar
from jupyter_cadquery import show, show_object, open_viewer, set_defaults

cv = open_viewer('CadQuery')

top_position = (192+90, 220, 300)
top_target = (192+90, 220, 0)
top_zoom = 1
    
set_defaults(
    #viewer=None,
    cad_width=800, 
    height=800, 
    tree_width=250,
    axes=False,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = top_position,
    target = top_target,
    zoom = top_zoom,
    
    #target=(0,0,200),
    #tools=False,
    glass=True,
    show_parent=False,
    #timeit=False
)

Overwriting auto display for cadquery Workplane and Shape


# Wing Construction

### Wing Configuration

In [3]:
from Airplane.aircraft_topology.WingConfiguration import WingConfiguration

airfoil = "../components/airfoils/naca2415.dat"
wing_config = WingConfiguration(root_airfoil=airfoil,
                  nose_pnt=(192.113, 0, -44.5),
                  root_chord=183,
                  root_dihedral=3.7,
                  root_incidence=3,
                  length=410,
                  sweep=20,
                  tip_chord=143,
                  tip_dihedral=0,
                  tip_incidence=0)
wing_config.add_segment(length=100,
                       sweep=20,
                       tip_chord=183-20,
                       tip_dihedral=15,
                       tip_incidence=0)
wing_config.add_segment(length=5,
                       sweep=5,
                       tip_chord=183-20-5,
                       tip_dihedral=15,
                       tip_incidence=0)
wing_config.add_segment(length=5,
                       sweep=5,
                       tip_chord=183-20-10,
                       tip_dihedral=15,
                       tip_incidence=0)
wing_config.add_segment(length=5,
                       sweep=5,
                       tip_chord=183-20-15,
                       tip_dihedral=15,
                       tip_incidence=0)
wing_config.add_segment(length=50,
                       sweep=45+50,
                       tip_chord=183-20-20-40-50,
                       tip_dihedral=0,
                       tip_incidence=0,
                       tip_airfoil="../components/airfoils/nacam2.dat")
wing_configuration = {"main_wing": wing_config}


## VaseModeRibCutoutCreator

In [4]:
import logging
from typing import Union, Literal

from cadquery import Workplane

from Airplane.AbstractShapeCreator import AbstractShapeCreator
from Airplane.aircraft_topology.WingConfiguration import WingConfiguration
from cq_plugins.wing.wing_segment import wing_segment
from cq_plugins.wing.wing_root_segment import wing_root_segment
from cq_plugins.fix_shape.fix_shape import fix_shape

class VaseModeRibCutoutCreator(AbstractShapeCreator):
    """
    Create a cutout shape that should be intersected with the hull shape.
    The shape :
               |   /||\      |
               |  / ||   \   |
    leading    | /  ||     \ | trailing
    edge       | \  ||     / | edge
               |  \ ||   /   |
               |   \||/      |
               offset      offest
    """
    def __init__(self, creator_id: str,
                 wing_index: Union[str, int],
                 printer_wall_thickness: float,
                 spare_support_geometry_is_round: bool,
                 spare_support_dimension_width: float,
                 spare_support_dimension_height: float,
                 leading_edge_offset: float,
                 trailing_edge_offset: float,
                 minimum_rib_angle: float,
                 wing_config: dict[int, WingConfiguration] = None,
                 wing_side: Literal["LEFT","RIGHT","BOTH"] = "RIGHT",
                 loglevel=logging.INFO):
        """
        parameters:
        printer_wall_thickness - printer settings wall thickness
        spare_support_geometry_is_round -- default true
        spare_support_dimension_x -- diameter if round is true
        spare_support_dimension_z -- ignored if round
        leading_edge_offset --
        trailing_edge_offset --
        minimum_rib_angle -- important for printability (should be > 45°)
        """
        self.printer_wall_thickness: float = printer_wall_thickness
        self.spare_support_geometry_is_round: bool = spare_support_geometry_is_round
        self.spare_support_dimension_width: float = spare_support_dimension_width
        self.spare_support_dimension_height: float = spare_support_dimension_height
        self.leading_edge_offset: float = leading_edge_offset
        self.trailing_edge_offset: float = trailing_edge_offset
        self.minimum_rib_angle: float = minimum_rib_angle
        self.wing_side: Literal["LEFT","RIGHT","BOTH"]  = wing_side
        self.wing_index: Union[str, int] = wing_index
        self._wing_config: dict[int, WingConfiguration] = wing_config

        super().__init__(creator_id, shapes_of_interest_keys=[], loglevel=loglevel)

    def _create_shape(self, shapes_of_interest: dict[str, Workplane],
                      input_shapes: dict[str, Workplane],
                      **kwargs) -> dict[str, Workplane]:
        logging.info(f"wing rib cutout from configuration --> '{self.identifier}'")

        wing_config: WingConfiguration = self._wing_config[self.wing_index]
        right_wing: Workplane = (
            Workplane('XZ')
            .wing_root_segment(
                root_airfoil=wing_config.segments[0].root_airfoil,
                root_chord=wing_config.segments[0].root_chord,
                root_dihedral=wing_config.segments[0].root_dihedral,
                root_incidence=wing_config.segments[0].root_incidence,
                length=wing_config.segments[0].length,
                sweep=wing_config.segments[0].sweep,
                tip_chord=wing_config.segments[0].tip_chord,
                tip_dihedral=wing_config.segments[0].tip_dihedral,
                tip_incidence=wing_config.segments[0].tip_incidence,
                tip_airfoil=wing_config.segments[0].tip_airfoil,
                offset=0))
        
        #for segment_config in wing_config.segments[1:]:
        #    right_wing: Workplane = (
        #        right_wing.wing_segment(
        #            length=segment_config.length,
        #            sweep=segment_config.sweep,
        #            tip_chord=segment_config.tip_chord,
        #            tip_dihedral=segment_config.tip_dihedral,
        #            tip_incidence=segment_config.tip_incidence,
        #            tip_airfoil=segment_config.tip_airfoil,
        #            offset=0))

        bb_right = right_wing.findSolid().BoundingBox(tolerance=1e-3)
        right_wing = right_wing.translate((0,-abs(bb_right.ymin)-1, 0))
        right_wing = right_wing.fix_shape()

        if self.wing_side == "LEFT":
            right_wing = right_wing.mirror("XZ")
        elif self.wing_side == "BOTH":
            left_wing = right_wing.mirror("XZ")
            right_wing = right_wing.union(left_wing)

        right_wing = right_wing.fix_shape()
        right_wing = right_wing.translate(wing_config.nose_pnt).display(name=f"{self.identifier}", severity=logging.DEBUG)

        return {self.identifier: right_wing}


In [5]:
printer_wall_thickness=0.42
spare_support_geometry_is_round=True
spare_support_dimension_width=6
spare_support_dimension_height=6
spare_position_factor=1/3 # value betweein (0,1) as fraction of the chord
leading_edge_offset=3
trailing_edge_offset=3
minimum_rib_angle=45

vm = VaseModeRibCutoutCreator(creator_id="wing",
                 wing_index="main_wing",
                 printer_wall_thickness=printer_wall_thickness,
                 spare_support_geometry_is_round=spare_support_geometry_is_round,
                 spare_support_dimension_width=spare_support_dimension_width,
                 spare_support_dimension_height=spare_support_dimension_height,
                 leading_edge_offset=leading_edge_offset,
                 trailing_edge_offset=trailing_edge_offset,
                 minimum_rib_angle=minimum_rib_angle,
                 wing_config=wing_configuration)

In [6]:
wing = vm._create_shape([],[])
wing['wing']

Camera control changed, so camera was resetted


## Vase Mode Rib Construction

Vase mode is a mode where the 3D printer does not need to change a layer, as layer changes are done contiously in spiral motion. To print like that, we need to construct a wing where everything can be printed in one motion, in each layer.

### Construction Workplane
First of all we need a workplane that is defined by the wing chord (including the angle of incidence) and the wing's dehedral.

In [7]:
import numpy as np
from scipy.spatial.transform import Rotation as R

# TODO: implement for all segments
# origin and angles need to be calculate over the whole kinematik chain.
def _wing_workplane(self, wing_config:WingConfiguration, segment:int = 0) -> Workplane:
    """
    Creating a workplan where the 0-point is located at the wing's nose point
    and the workplane is going through the wing.
    
    Remark: an incident angle at the wing_tip cannot be covered with this 
    workplane.
    """
    origin = wing_config.nose_pnt

    r = R.from_euler('y', wing_config.segments[segment].root_incidence, degrees=True)
    xdir = r.apply((1,0,0)) # x-unit vector rotate by root_incidence

    normal = r.apply((0,0,1)) # x-unit vector rotate by root_incidence
    r = R.from_euler('x', wing_config.segments[segment].root_dihedral, degrees=True)
    normal = r.apply(normal) # x-unit vector rotate by root_incidence

    plane = cq.Plane(origin=origin, xDir=xdir.tolist(), normal=normal.tolist())

    wp_plane = (cq.Workplane(inPlane=plane, origin=origin))
    
    return wp_plane

VaseModeRibCutoutCreator.wing_workplane = _wing_workplane

In [8]:
wing_wp = vm.wing_workplane(wing_configuration['main_wing'])

### Wing Trapezoid Calculation

We need to calculate the wing's trapezoid outlines as boundaries.

from typing import Tuple, TypeVar, Optional, cast as tcast
from multimethod import multimethod

from cadquery.occ_impl.shapes import Shape, Face, Edge, Wire, Compound, Vertex, edgesToWires
from cadquery.occ_impl.geom import Location, Vector

T = TypeVar("T", bound="Sketch")
Modes = Literal["a", "s", "i", "c"]  # add, subtract, intersect, construct
Point = Union[Vector, Tuple[float, float]]

def _line(p1, p2):
    A = (p1[1] - p2[1])
    B = (p2[0] - p1[0])
    C = (p1[0]*p2[1] - p2[0]*p1[1])
    return A, B, -C

def _intersection(L1, L2):
    D  = L1[0] * L2[1] - L1[1] * L2[0]
    Dx = L1[2] * L2[1] - L1[1] * L2[2]
    Dy = L1[0] * L2[2] - L1[2] * L2[0]
    if D != 0:
        x = Dx / D
        y = Dy / D
        return x,y
    else:
        return False
    
def _segmentToIntersection(edge: Edge, v1: Point, v2: Point) -> Edge:
        s1 = edge.startPoint()
        s2 = edge.endPoint()
        L1 = _line([v1.x, v1.y], [v2.x, v2.y])
        L2 = _line([s1.x, s1.y], [s2.x, s2.y])
        R = _intersection(L1, L2)
        if (R == False):
            return False
        return Edge.makeLine(v1, Vector(R))
    

@multimethod
def segmentToSegment(self: T, point: Point, direction: Point, end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the given point.
    
    point.   : start point
    start_tag: start edge tag
    direction: direction of the segment 
    end_tag  : the finale edge to reach
    """
    
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    v2: Vector = Vector(direction)*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

@segmentToSegment.register
def segmentToSegment(self: T, point: Point, angle: Union[float, int], end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the given point.
    
    point.   : start point
    start_tag: start edge tag
    angle    : direction of the segment as angle in degrees from positive x
    end_tag  : the finale edge to reach
    """
    
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    r = R.from_euler('z', angle, degrees=True)
    direction = Vector(tuple(r.apply((10.,0.,0.)))) # x-unit vector rotate by root_incidence
    v2: Vector = Vector(direction)*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self 
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

@segmentToSegment.register
def segmentToSegment(
    self: T, direction: Point, end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the end point of the last edge.
    
    direction: direction vector of the segment
    end_tag  : the finale edge to reach
    """
    
    point = self._endPoint()
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    v2: Vector = direction*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

@segmentToSegment.register
def segmentToSegment(
    self: T, start_tag: str, direction: Point, end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the end point of the tagged start edge.
    
    start_tag: start edge tag
    direction: direction of the segment 
    end_tag  : the finale edge to reach
    """
    
    seg1 = self._tags[start_tag][0]
    point = tcast(Edge, seg1).endPoint()
    
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    v2: Vector = direction*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

@segmentToSegment.register
def segmentToSegment(
    self: T, angle: Union[float, int], end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the end point of the last edge.
    
    angle    : direction of the segment as angle in degrees from positive x
    end_tag  : the finale edge to reach
    """
    
    point = self._endPoint()
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    r = R.from_euler('z', angle, degrees=True)
    direction = Vector(tuple(r.apply((10.,0.,0.)))) # x-unit vector rotate by root_incidence
    v2: Vector = direction*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

@segmentToSegment.register
def segmentToSegment(
    self: T, start_tag: str, angle: Union[float, int], end_tag: str, tag: Optional[str] = None,forConstruction: bool = False
) -> T:
    """
    Construction of a segment that stops at the given tagged end edge. 
    Starting at the end point of the tagged start edge.
    
    start_tag: start edge tag
    angle    : direction of the segment as angle in degrees from positive x
    end_tag  : the finale edge to reach
    """
    
    seg1 = self._tags[start_tag][0]
    point = tcast(Edge, seg1).endPoint()
    
    seg2 = self._tags[end_tag][0]
    edge = tcast(Edge, seg2)
    v1: Vector = Vector(point)
    r = R.from_euler('z', angle, degrees=True)
    direction = Vector(tuple(r.apply((10.,0.,0.)))) # x-unit vector rotate by root_incidence
    v2: Vector = direction*10 + v1

    # dispatch on geom type
    if edge.geomType() == "LINE":
        val = _segmentToIntersection(edge, v1, v2)
        if (val == False):
            return self
    #elif v0.geomType() == "CIRCLE":   
    else:
        return self
        
    return self.edge(val, tag, forConstruction)

cq.Sketch.segmentToSegment = segmentToSegment

In [226]:
set_defaults(
    axes=False,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (90,200,300),
    target = (90,200,0),
)

In [227]:
from math import sqrt
segment = 0
root_nose = np.asarray((.0,.0, .0))
root_tail = np.asarray((1.,.0, .0)) * wing_config.segments[segment].root_chord
tip_nose = np.asarray((wing_config.segments[segment].sweep, sqrt(wing_config.segments[segment].sweep**2 + wing_config.segments[segment].length**2),0))
tip_tail = tip_nose + np.asarray((1.,.0, .0)) * wing_config.segments[segment].tip_chord

Calculating the spare nose and tail positions

In [228]:
spare_nose_root = ( np.asarray((1.,0.,0.)) * wing_config.segments[segment].root_chord * spare_position_factor 
                   - np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2  
                   - np.asarray((1.,0.,0.)) *  printer_wall_thickness )
spare_nose_tip = ( tip_nose + np.asarray((1.,0.,0.)) * wing_config.segments[segment].tip_chord*spare_position_factor 
                  - np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2 
                  - np.asarray((1.,0.,0.)) *  printer_wall_thickness )

In [229]:
spare_tail_root = ( np.asarray((1.,0.,0.)) * wing_config.segments[segment].root_chord * spare_position_factor 
                   + np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2  
                   + np.asarray((1.,0.,0.)) *  printer_wall_thickness )
spare_tail_tip = ( tip_nose + np.asarray((1.,0.,0.)) * wing_config.segments[segment].tip_chord*spare_position_factor 
                  + np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2 
                  + np.asarray((1.,0.,0.)) *  printer_wall_thickness )

In [230]:
nose_bottom_lp =  root_nose + np.asarray((1.,0.,0.)) * leading_edge_offset
tail_bottom_lp = root_tail + np.asarray((1.,0.,0.)) * -trailing_edge_offset

r = R.from_euler('z', -45, degrees=True)
nose_top_lp_dir = r.apply((0.,100.,0.)) # x-unit vector rotate by root_incidence

#tail_top_lp =

In [238]:
from cq_plugins.sketch.segmentToEdge import segmentToEdge
from cq_plugins.fix_shape.fix_shape import fix_shape


const_lines = (cq.Sketch()
                .segment(Vector(tuple(root_nose)), 
                        Vector(tuple(root_tail)),'root')
                .segment(Vector(tuple(root_nose)), 
                        Vector(tuple(tip_nose)),'nose', forConstruction=True)
                .segment(Vector(tuple(root_tail)), 
                        Vector(tuple(tip_tail)),'tail', forConstruction=True)
                .segment(Vector(tuple(spare_tail_root)), 
                        Vector(tuple(spare_tail_tip)),'spare_tail', forConstruction=True)
                .segment(Vector(tuple(spare_nose_root)), 
                        Vector(tuple(spare_nose_tip)),'spare_nose', forConstruction=True)
                .segmentToEdge(Vector(tuple(nose_bottom_lp)),
                                  Vector(tuple(nose_top_lp_dir)),
                                  'spare_nose', 'rib')
                .segmentToEdge(90+45,
                                  'nose', 'rib__')
                .segmentToEdge(Vector(tuple(nose_bottom_lp)),
                                  45,
                                  'spare_nose', 'rib')
               .segmentToEdge(Vector((1,0,0)),
                                 'spare_tail', forConstruction=True)
                .segment(Vector(tuple(tail_bottom_lp)),'rib_')
                .segmentToEdge('rib__', 45,
                                  'spare_nose', 'rib___')
               .clean()
              )
const_lines

TypeError: 'module' object is not callable

In [198]:
const_lines.edges('+X').delete()